# メル値 (.npy) を開く

`result/mel/` に XCUITest が書き出した `.npy` ファイルを読み込んで中身を確認する。

起動方法 (リポジトリルートで):
```bash
PronounSE/venv/bin/jupyter notebook notebooks/explore_mel.ipynb
```

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

MEL_DIR = os.path.join('..', 'result', 'mel')

# 利用可能な .npy ファイルを列挙
files = sorted([f for f in os.listdir(MEL_DIR) if f.endswith('.npy')])
for f in files:
    print(f)

## 1つ読み込んで中身を見る

In [ ]:
mel = np.load(os.path.join(MEL_DIR, 'output_mel_Float16_cpuOnly.npy'))
print(f'shape: {mel.shape}  (frames × n_mels)')
print(f'dtype: {mel.dtype}')
print(f'min={mel.min():.2f} max={mel.max():.2f} mean={mel.mean():.2f}  (単位: dB)')
mel  # ノートブック末尾で値を表示

## ヒートマップで描画

In [ ]:
def plot_mel(mel, title='', ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 3))
    # mel[t, b] → 表示は縦軸 mel bin (低周波が下), 横軸 time
    im = ax.imshow(mel.T, origin='lower', aspect='auto', cmap='magma', vmin=-80, vmax=0)
    ax.set_xlabel('frame')
    ax.set_ylabel('mel bin')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='dB')
    return ax

plot_mel(mel, 'output_mel_Float16_cpuOnly')
plt.tight_layout()
plt.show()

## 全12通りを1枚に並べて比較

In [ ]:
precisions = ['Float32', 'Float16', 'Int8']
devices    = ['cpuOnly', 'cpuAndGPU', 'cpuAndNE', 'all']

fig, axes = plt.subplots(len(precisions), len(devices), figsize=(16, 7), sharex=True, sharey=True)
for i, prec in enumerate(precisions):
    for j, dev in enumerate(devices):
        arr = np.load(os.path.join(MEL_DIR, f'output_mel_{prec}_{dev}.npy'))
        ax = axes[i, j]
        ax.imshow(arr.T, origin='lower', aspect='auto', cmap='magma', vmin=-80, vmax=0)
        ax.set_title(f'{prec} / {dev}', fontsize=9)
        if i == len(precisions) - 1:
            ax.set_xlabel('frame')
        if j == 0:
            ax.set_ylabel('mel bin')
plt.tight_layout()
plt.show()

## cpuOnly を基準にして差分を可視化

赤 = GPU/ANE 側が大きい、青 = 小さい。真っ白に近いほど差が小さい。

In [ ]:
fig, axes = plt.subplots(len(precisions), len(devices) - 1, figsize=(14, 7), sharex=True, sharey=True)
for i, prec in enumerate(precisions):
    base = np.load(os.path.join(MEL_DIR, f'output_mel_{prec}_cpuOnly.npy'))
    for j, dev in enumerate([d for d in devices if d != 'cpuOnly']):
        arr = np.load(os.path.join(MEL_DIR, f'output_mel_{prec}_{dev}.npy'))
        diff = arr - base
        vmax = max(abs(diff.min()), abs(diff.max()), 1e-6)
        ax = axes[i, j]
        im = ax.imshow(diff.T, origin='lower', aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f'{prec}: {dev} - cpuOnly  (max|d|={vmax:.2g})', fontsize=9)
        plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()